### Simple CNN Image Segmentation Model Training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importing Necessary Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
base_path = r'/content/drive/MyDrive/COCO2017' # dataset base path
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Hyperparameters
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-3
IMG_SIZE = 256

In [ ]:
class COCOSegmentationDataset(Dataset):
    """
    Custom Dataset for COCO segmentation task.
    Loads paired RGB images and binary masks from COCO train/val splits.
    """
    def __init__(self, split='train', img_size=IMG_SIZE, transform=None):
        """
        Initialize dataset for specific split (train/val).

        Args:
            split (str): 'train' or 'val' dataset split
            img_size (int): Target image size for resizing (square)
            transform (callable): Optional transform for images (not masks)
        """
        self.split = split
        self.img_size = img_size
        self.transform = transform

        # Construct paths to mask directory (created by mask generation script)
        masks_dir = os.path.join(base_path, f"mask/masks_{split}")
        self.mask_files = [f for f in os.listdir(masks_dir) if f.endswith('.png')]
        self.masks_dir = masks_dir

        # Derive original image filenames from mask filenames
        # masks_{split}/COCO_train2017_000000000009_mask.png -> COCO_train2017_000000000009.jpg
        self.img_names = [f.replace('_mask.png', '.jpg') for f in self.mask_files]

        # Path to original COCO images
        self.img_dir = os.path.join(base_path, 'raw',
                                   'train2017' if split == 'train' else 'val2017')

        print(f"Loaded {len(self.mask_files)} {split} samples")

    def __len__(self):
        """Return total number of samples in dataset."""
        return len(self.mask_files)

    def __getitem__(self, idx):
        """
        Load and preprocess single image-mask pair at index idx.

        Returns:
            tuple: (image_tensor: [C,H,W], mask_tensor: [1,H,W])
        """
        # Load corresponding image
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')  # Ensure 3-channel RGB

        # Load corresponding binary mask (grayscale PNG)
        mask_name = self.mask_files[idx]
        mask_path = os.path.join(self.masks_dir, mask_name)
        mask = Image.open(mask_path).convert('L')    # Load as single-channel grayscale

        # Resize both to consistent size (maintains aspect ratio via PIL)
        image = image.resize((self.img_size, self.img_size))
        mask = mask.resize((self.img_size, self.img_size))

        # Convert PIL Images to numpy arrays
        # Image: HWC [0-255] -> CHW [0-1.0] float
        image = np.array(image).transpose(2, 0, 1) / 255.0

        # Mask: HW [0-255] -> HW [0-1.0] float (binary)
        mask = np.array(mask) / 255.0

        # Convert to PyTorch tensors
        image = torch.FloatTensor(image)  # Shape: [3, H, W]
        mask = torch.FloatTensor(mask)    # Shape: [H, W]

        # Apply optional data augmentation (only to images)
        if self.transform:
            image = self.transform(image)

        # Add channel dimension to mask for segmentation: [1, H, W]
        return image, mask.unsqueeze(0)


In [ ]:
class SimpleCNN(nn.Module):
    """
    Simple CNN for binary image segmentation.
    Encoder-decoder architecture
    Input: [B, 3, 256, 256] -> Output: [B, 1, 256, 256]
    Total params: ~5.2M (lightweight)
    """
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleCNN, self).__init__()

        # === ENCODER: Progressive downsampling (feature extraction) ===
        # Each block: 2x Conv3x3 + ReLU + MaxPool2 (halves spatial dims)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),   # 3->32, 256x256
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),           # 32->32
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)                             # 256x256 -> 128x128
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),            # 32->64, 128x128
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),            # 64->64
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)                             # 128x128 -> 64x64
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),           # 64->128, 64x64
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),          # 128->128
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)                             # 64x64 -> 32x32
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),          # 128->256, 32x32
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),          # 256->256 (bottleneck)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)                             # 32x32 -> 16x16
        )

        # === DECODER: Progressive upsampling with skip connections ===

        # Upsample + concat with encoder features + refine
        self.upconv4 = nn.ConvTranspose2d(256, 128, 2, stride=2)   # 16x16 -> 32x32
        self.dec4 = nn.Sequential(                                 # Skip: 256+128=384? Wait, see forward
            nn.Conv2d(256, 128, 3, padding=1),                     # Refine concatenated features
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv3 = nn.ConvTranspose2d(128, 64, 2, stride=2)   # 32x32 -> 64x64
        self.dec3 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),                     # Skip: 128+64=192
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv2 = nn.ConvTranspose2d(64, 32, 2, stride=2)    # 64x64 -> 128x128
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),                      # Skip: 64+32=96
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv1 = nn.ConvTranspose2d(32, 16, 2, stride=2)    # 128x128 -> 256x256
        self.final_conv = nn.Sequential(
            nn.Conv2d(48, 32, 3, padding=1),                      # Skip: 16+3(input)=19? See forward note
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_channels, 1),                       # 1x1 conv for pixel-wise prediction
            nn.Sigmoid()                                          # Binary output [0,1]
        )

    def forward(self, x):
        """
        Forward pass through encoder-decoder.

        Args:
            x: Input [B, 3, 256, 256]
        Returns:
            Segmentation mask [B, 1, 256, 256]
        """
        # === ENCODER PATH === Save skip connections
        e1 = self.conv1(x)      # [B, 32, 128, 128]
        e2 = self.conv2(e1)     # [B, 64, 64, 64]
        e3 = self.conv3(e2)     # [B, 128, 32, 32]
        e4 = self.conv4(e3)     # [B, 256, 16, 16] (bottleneck)

        # === DECODER PATH === Upsample + Skip Connections
        d4 = self.upconv4(e4)   # [B, 128, 32, 32]
        d4 = torch.cat([d4, e3], dim=1)  # [B, 256, 32, 32] (128+128)
        d4 = self.dec4(d4)      # [B, 128, 32, 32]

        d3 = self.upconv3(d4)   # [B, 64, 64, 64]
        d3 = torch.cat([d3, e2], dim=1)   # [B, 128, 64, 64] (64+64)
        d3 = self.dec3(d3)      # [B, 64, 64, 64]

        d2 = self.upconv2(d3)   # [B, 32, 128, 128]
        d2 = torch.cat([d2, e1], dim=1)   # [B, 64, 128, 128] (32+32)
        d2 = self.dec2(d2)      # [B, 32, 128, 128]

        d1 = self.upconv1(d2)   # [B, 16, 256, 256]
        d1 = torch.cat([d1, x], dim=1)    # [B, 19, 256, 256] (16+3) ← Input skip!
        out = self.final_conv(d1)         # [B, 1, 256, 256]

        return out


- Model Architecture

Input (3×256×256) → Encoder (32→64→128→256) → Bottleneck (256×16×16)
→ Decoder (256→128→64→32→16) + 4 Skip Connections → Output (1×256×256)

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice Loss for binary segmentation.
    1 - Dice Coefficient, optimized for imbalanced foreground/background.
    Formula: 1 - (2 * |P∩G| + smooth) / (|P| + |G| + smooth)
    """
    def __init__(self, smooth=1):
        """
        Args:
            smooth (float): Smoothing factor to avoid division by zero
                           Common values: 1.0 (default) or 0.0001
        """
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        """
        Args:
            pred: Model predictions [B, 1, H, W] in range [0,1]
            target: Ground truth masks [B, 1, H, W] binary [0,1]

        Returns:
            dice_loss: Scalar loss value [0,1] (1=perfect overlap, 0=no overlap)
        """
        # Flatten all spatial dimensions for coefficient computation
        # [B,1,H,W] -> [B*1*H*W] ensures contiguous memory layout
        pred = pred.contiguous().view(-1)      # Predicted pixels (soft)
        target = target.contiguous().view(-1)  # Ground truth pixels (binary)

        # Compute intersection: sum(pred * target) over all pixels
        intersection = (pred * target).sum()

        # Dice coefficient: 2*intersection / (pred_sum + target_sum + smooth)
        dice = (2. * intersection + self.smooth) / \
               (pred.sum() + target.sum() + self.smooth)

        # Loss = 1 - Dice (minimize → maximize overlap)
        return 1 - dice


def iou_score(pred, target, threshold=0.5):
    """
    Intersection over Union (IoU/Jaccard) metric for segmentation.
    IoU = |P∩G| / |P∪G| = intersection / (pred_sum + target_sum - intersection)

    Args:
        pred: Model predictions [B, 1, H, W] in range [0,1]
        target: Ground truth masks [B, 1, H, W] binary [0,1]
        threshold (float): Binarization threshold for predictions

    Returns:
        iou: Scalar IoU score [0,1] (1=perfect overlap)
    """
    # Binarize predictions using threshold (>0.5 → foreground)
    pred = (pred > threshold).float()        # Soft → Hard binary predictions

    # Binarize target masks (already ~0/1, threshold ensures clean binary)
    target = (target > 0.5).float()          # Ensure strict binary ground truth

    # Intersection: overlapping foreground pixels
    intersection = (pred * target).sum()

    # Union: |P| + |G| - |P∩G|
    union = pred.sum() + target.sum() - intersection

    # IoU with epsilon to avoid div-by-zero (empty predictions)
    return (intersection + 1e-6) / (union + 1e-6)


In [ ]:
## Data augmentation pipeline for training robustness.

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),        # 50% chance horizontal flip
    transforms.RandomRotation(10),                  # ±10° random rotation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # ±20% brightness/contrast
    # Note: No RandomCrop/Resize here - already handled in dataset __getitem__
])

# === DATASET INSTANTIATION ===
"""
Create train/val datasets.
- train: With augmentation for better generalization
- val: No augmentation for clean evaluation
"""
train_dataset = COCOSegmentationDataset(
    split='train',
    transform=train_transform  # Augmentation applied in __getitem__
)

val_dataset = COCOSegmentationDataset(
    split='val',
    transform=None  # No augmentation for validation
)

# === DATA LOADERS: Parallel loading for speed ===
"""
DataLoader configuration optimized for GPU training.
- batch_size: Controls memory usage (16 fits ~11GB VRAM)
- shuffle: Randomize training order each epoch
- num_workers: Parallel image loading (2-4 recommended for Colab)
"""
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,      # e.g., 16 images per batch
    shuffle=True,               # Randomize training batches each epoch
    num_workers=2,              # Parallel data loading
    pin_memory=True             # Faster CPU→GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,      # Same batch size for consistent metrics
    shuffle=False,              # Sequential order for validation
    num_workers=2,              # Parallel loading
    pin_memory=True
)

print(f"✓ Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"✓ Val:   {len(val_dataset)} samples, {len(val_loader)} batches")


Epoch Loop:

├── Train: model.train() → DataLoader → Forward/Backward → Metrics

├── Val:   model.eval() → no_grad() → Forward Only → Metrics

├── Log:   Average metrics → Print → Scheduler → Save best

└── Repeat for EPOCHS

In [ ]:
# === MODEL & OPTIMIZER SETUP ===
"""
Initialize model, optimizer, scheduler, and loss function.
Move model to GPU for accelerated training.
"""
model = SimpleCNN().to(device)  # ~5.2M params, encoder-decoder architecture
optimizer = optim.Adam(model.parameters(), lr=LR)  # Adam optimizer with 1e-3 LR
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3  # Reduce LR if val loss doesn't improve for 3 epochs
)
criterion = DiceLoss()  # Dice loss optimized for segmentation imbalance

# === TRAINING HISTORY STORAGE ===
train_losses, val_losses = [], []    # Per-epoch average losses
train_ious, val_ious = [], []        # Per-epoch average IoU scores

print("Starting training...")
for epoch in range(EPOCHS):
    # === TRAINING PHASE ===
    model.train()  # Enable dropout/batchnorm training mode
    train_loss, train_iou = 0, 0  # Accumulators for epoch averages

    # Progress bar for training batches
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')

    for batch_idx, (images, masks) in enumerate(train_pbar):
        # Move batch to GPU (non-blocking if pin_memory=True)
        images, masks = images.to(device), masks.to(device)

        # Forward-backward pass
        optimizer.zero_grad()           # Clear gradients
        preds = model(images)           # Forward pass [B,1,H,W]
        loss = criterion(preds, masks)  # Compute Dice loss
        loss.backward()                 # Backprop
        optimizer.step()                # Update weights

        # Accumulate metrics for epoch average
        train_loss += loss.item()
        train_iou += iou_score(preds, masks).item()

        # Live progress display (current batch metrics)
        train_pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'IoU': f'{iou_score(preds, masks).item():.4f}'
        })

    # === VALIDATION PHASE ===
    model.eval()  # Disable dropout/batchnorm, enable inference mode
    val_loss, val_iou = 0, 0

    with torch.no_grad():  # Disable gradient computation (saves memory/speed)
        val_pbar = tqdm(val_loader, desc='Val', leave=False)  # Nested progress bar
        for images, masks in val_pbar:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss = criterion(preds, masks)

            val_loss += loss.item()
            val_iou += iou_score(preds, masks).item()

    # === EPOCH METRICS (AVERAGES) ===
    # Compute mean metrics across all batches
    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))
    train_ious.append(train_iou / len(train_loader))
    val_ious.append(val_iou / len(val_loader))

    # Print epoch summary
    print(f'Epoch {epoch+1}: '
          f'Train Loss={train_losses[-1]:.4f}, IoU={train_ious[-1]:.4f} | '
          f'Val Loss={val_losses[-1]:.4f}, IoU={val_ious[-1]:.4f}')

    # Learning rate scheduling based on validation loss
    scheduler.step(val_losses[-1])

    # === MODEL CHECKPOINTING ===
    # Save best model based on validation IoU (highest is best)
    if len(val_ious) == 1 or val_ious[-1] > max(val_ious[:-1]):
        torch.save(model.state_dict(), f'coco_simplecnn_best_{IMG_SIZE}.pth')
        print(f"✓ New best model saved (Val IoU: {val_ious[-1]:.4f})")

print("✓ Training completed! Best model saved as 'coco_simplecnn_best_256.pth'")